# Our text generation v1 — Kaggle
Sinh caption cho **source, target, adversarial** từ output đã giải nén của `test_attack_v1.ipynb`.

Bật Internet; bật GPU cho LLaVA/InternVL/Qwen. API dùng Kaggle Secrets `OPENAI_API_KEY` hoặc `GEMINI_API_KEY` (có phí). Chạy lần lượt các cell; mặc định kiểm tra cấu hình với `RUN_CAPTION=False`. Sau khi kiểm tra input, đặt `RUN_CAPTION=True` và chạy lại từ Cell 1.

Protocol v1 giữ nguyên prompt và tham số trong repo. Class được chép nguyên văn từ `src/FOA-Attack/blackbox_text_generation.py`. Source được clone/pull theo GIT_BRANCH; SHA thực tế chỉ được ghi lại trong output, không khóa phiên bản Git. Code nối manifest/xuất file nằm ở mức top-level, không định nghĩa hàm mới. Caption có thể vượt 20 từ nếu model không tuân thủ; notebook giữ nguyên văn phản hồi, không cắt/sửa.

Mỗi lần chạy tạo thư mục mới. ZIP caption không kèm ảnh; giữ ZIP attack riêng. Captioning và attack nên chạy trong session riêng do khác phiên bản Transformers.


In [ ]:
# Cell 1 — Chọn model và thiết lập vận hành tại đây.
from pathlib import Path

MODEL = "LLaVA-1.5-7B"
MODEL_CONFIGS = {
    "GPT-5 mini": "caption_gpt5mini_v1",
    "Gemini 2.5 Flash": "caption_gemini25flash_v1",
    "LLaVA-1.5-7B": "caption_llava15_7b_v1",
    "InternVL3-8B": "caption_internvl3_8b_v1",
    "Qwen3-VL-8B": "caption_qwen3vl_8b_v1",
}
INPUT_DIR = Path("/kaggle/input/foa-attack-output")
# Cùng session attack: INPUT_DIR = Path("/kaggle/working/foa_attack_output")
OUTPUT_DIR = Path("/kaggle/working/our_text_generation_outputs")
NUM_SAMPLES = 1          # None: toàn bộ manifest; 1: smoke test.
DEVICE = "cuda:0"        # Implementation tác giả cố định GPU 0; không hỗ trợ đổi GPU.
RUN_CAPTION = False
REPO_URL = "https://github.com/zintomvn/Code-Adversarial-Attack-on-VLM.git"
GIT_BRANCH = "main"
REPO_DIR = Path("/kaggle/working/Code-Adversarial-Attack-on-VLM")

assert MODEL in MODEL_CONFIGS, f"Chọn model trong {tuple(MODEL_CONFIGS)}"
assert DEVICE == "cuda:0", "Class gốc cố định cuda:0."
assert NUM_SAMPLES is None or (type(NUM_SAMPLES) is int and NUM_SAMPLES > 0)
CONFIG_NAME = MODEL_CONFIGS[MODEL]
LOCAL_MODEL = MODEL in {"LLaVA-1.5-7B", "InternVL3-8B", "Qwen3-VL-8B"}
assert INPUT_DIR.is_dir(), f"Add Data và chỉnh INPUT_DIR: {INPUT_DIR}"
manifest_candidates = sorted(INPUT_DIR.rglob("manifest.jsonl"))
if len(manifest_candidates) != 1:
    raise ValueError(f"Cần đúng một manifest.jsonl; tìm thấy {len(manifest_candidates)}. Chọn thư mục của một run.")
MANIFEST_PATH = manifest_candidates[0]
INPUT_ROOT = MANIFEST_PATH.parent.resolve()
print("Model:", MODEL, "| Config:", CONFIG_NAME)
print("Manifest:", MANIFEST_PATH, "| Run:", RUN_CAPTION)


In [ ]:
# Cell 2 — Exact dependency pins từ author configs; giữ Torch/Torchvision của Kaggle.
import subprocess
import sys

PACKAGES = [
    "hydra-core==1.3.2", "omegaconf==2.3.0", "PyYAML==6.0.2",
    "wandb==0.19.1", "tenacity==9.0.0", "openai==1.109.1",
    "google-genai==1.41.0", "anthropic==0.45.2",
]
if LOCAL_MODEL:
    PACKAGES += [
        "transformers==4.57.6", "accelerate==1.10.1",
        "bitsandbytes==0.48.1", "sentencepiece==0.2.1",
    ]
subprocess.run([sys.executable, "-m", "pip", "install", *PACKAGES], check=True)
print("Dependencies installed. Nếu package đã được import trước pip, restart session rồi chạy lại.")


In [ ]:
# Cell 3 — Clone/pull branch như notebook hiện có; ghi lại SHA thực tế.
import os
import hashlib
import json

REFERENCE_IMPLEMENTATION = {
    "repository": REPO_URL,
    "branch": GIT_BRANCH,
}
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", GIT_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    assert (REPO_DIR / ".git").is_dir(), f"Không phải Git repository: {REPO_DIR}"
    subprocess.run(["git", "-C", str(REPO_DIR), "switch", GIT_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", GIT_BRANCH], check=True)
RESOLVED_REVISION = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
SOURCE_DIR = REPO_DIR / "src" / "FOA-Attack"
assert (SOURCE_DIR / "config" / f"{CONFIG_NAME}.yaml").is_file()
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
os.environ.update({
    "HF_HOME": "/kaggle/working/our_caption_hf_cache",
    "WANDB_MODE": "disabled", "TOKENIZERS_PARALLELISM": "false",
})
print("Source branch:", GIT_BRANCH, "| Actual commit:", RESOLVED_REVISION)


In [ ]:
# Cell 4 — Compose config tác giả; bảo vệ protocol trước khi gọi API/tải weights.
from utils import KaggleRuntime, encode_image
from omegaconf import OmegaConf
import torch

assert Path(sys.modules["utils"].__file__).resolve() == (SOURCE_DIR / "utils.py").resolve()
cfg = KaggleRuntime.config(CONFIG_NAME)
EXPECTED_MODELS = {
    "GPT-5 mini": ("openai", "gpt-5-mini-2025-08-07", "none"),
    "Gemini 2.5 Flash": ("gemini", "gemini-2.5-flash", "none"),
    "LLaVA-1.5-7B": ("huggingface", "llava-hf/llava-1.5-7b-hf", "nf4"),
    "InternVL3-8B": ("huggingface", "OpenGVLab/InternVL3-8B-hf", "nf4"),
    "Qwen3-VL-8B": ("huggingface", "Qwen/Qwen3-VL-8B-Instruct", "nf4"),
}
assert (cfg.caption.provider, cfg.caption.model_id, cfg.caption.quantization) == EXPECTED_MODELS[MODEL]
assert cfg.caption.prompt == "Describe this image in one concise sentence, no longer than 20 words."
assert (cfg.caption.timeout, cfg.caption.reasoning_effort,
        cfg.caption.max_output_tokens, cfg.caption.max_new_tokens,
        cfg.caption.revision, cfg.experiment.seed) == (90, "minimal", 2048, 128, "main", 2023)
assert list(cfg.environment.packages) == PACKAGES
if LOCAL_MODEL:
    assert torch.cuda.is_available(), "Bật GPU trong Kaggle Settings."
    print("GPU:", torch.cuda.get_device_name(0))
print(OmegaConf.to_yaml(cfg.caption, resolve=True))


In [ ]:
# Cell 5 — Class tác giả chép nguyên văn; helpers/base class import từ repo.
from blackbox_text_generation import ImageDescriptionGenerator
from tenacity import retry, wait_random_exponential, stop_after_attempt
from openai import OpenAI
from google import genai

class VersionedImageDescriptionGenerator(ImageDescriptionGenerator):
    """Common caption prompt; modern adapters without changing legacy generators."""

    def __init__(self, cfg):
        self.cfg = cfg
        self.model_name = cfg.caption.model_id
        self.metadata = {}
        if cfg.caption.provider == 'openai':
            self.client = OpenAI(api_key=KaggleRuntime.secret('OPENAI_API_KEY'), timeout=cfg.caption.timeout, max_retries=3)
        elif cfg.caption.provider == 'gemini':
            from google.genai import types
            self.client = genai.Client(api_key=KaggleRuntime.secret('GEMINI_API_KEY'),
                                       http_options=types.HttpOptions(timeout=int(cfg.caption.timeout * 1000)))
        elif cfg.caption.provider == 'huggingface':
            self._load_local()
        else:
            raise ValueError(f'Unknown provider: {cfg.caption.provider}')

    def _load_local(self):
        from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
        if not torch.cuda.is_available():
            raise RuntimeError('Local VLM inference requires a Kaggle GPU.')
        torch.manual_seed(self.cfg.experiment.seed)
        dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
        options = dict(device_map={'': 0}, torch_dtype=dtype, low_cpu_mem_usage=True, attn_implementation='eager',
                       revision=self.cfg.caption.revision, trust_remote_code=False)
        if self.cfg.caption.quantization == 'nf4':
            options['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                                               bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
        elif self.cfg.caption.quantization != 'none':
            raise ValueError('Use quantization=nf4 or none.')
        self.processor = AutoProcessor.from_pretrained(self.model_name, revision=self.cfg.caption.revision, trust_remote_code=False)
        self.model = AutoModelForImageTextToText.from_pretrained(self.model_name, **options).eval()
        self.metadata = {'resolved_revision': getattr(self.model.config, '_commit_hash', None),
                         'dtype': str(dtype), 'quantization': self.cfg.caption.quantization,
                         'gpu': torch.cuda.get_device_name(0)}

    @retry(wait=wait_random_exponential(min=1, max=30), stop=stop_after_attempt(3), reraise=True)
    def generate_description(self, image_path):
        cfg = self.cfg.caption
        self.last_response = {}
        if cfg.provider == 'openai':
            response = self.client.responses.create(
                model=self.model_name, store=False, reasoning={'effort': cfg.reasoning_effort},
                max_output_tokens=cfg.max_output_tokens,
                input=[{'role': 'user', 'content': [
                    {'type': 'input_text', 'text': cfg.prompt},
                    {'type': 'input_image', 'image_url': f'data:image/png;base64,{encode_image(image_path)}', 'detail': 'high'},
                ]}])
            if response.status != 'completed':
                raise ValueError(f'Incomplete OpenAI response: {response.status}')
            text = response.output_text
            self.last_response = {'response_id': response.id, 'model': response.model,
                                  'usage': response.usage.model_dump(mode='json') if response.usage else None}
        elif cfg.provider == 'gemini':
            from google.genai import types
            with open(image_path, 'rb') as stream:
                part = types.Part.from_bytes(data=stream.read(), mime_type='image/png')
            response = self.client.models.generate_content(model=self.model_name, contents=[cfg.prompt, part],
                config=types.GenerateContentConfig(temperature=0, max_output_tokens=cfg.max_output_tokens,
                                                   thinking_config=types.ThinkingConfig(thinking_budget=0)))
            if not response.candidates or 'STOP' not in str(response.candidates[0].finish_reason):
                raise ValueError('Gemini output blocked or truncated.')
            text = response.text
            self.last_response = {'model': getattr(response, 'model_version', self.model_name),
                                  'usage': response.usage_metadata.model_dump(mode='json') if response.usage_metadata else None}
        else:
            messages = [{'role': 'user', 'content': [{'type': 'image', 'url': os.path.abspath(image_path)},
                                                    {'type': 'text', 'text': cfg.prompt}]}]
            inputs = self.processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                                         return_dict=True, return_tensors='pt')
            inputs = inputs.to(self.model.device, dtype=self.model.dtype)
            with torch.inference_mode():
                tokens = self.model.generate(**inputs, do_sample=False, max_new_tokens=cfg.max_new_tokens)
            generated = tokens[0, inputs['input_ids'].shape[1]:]
            if len(generated) >= cfg.max_new_tokens:
                raise ValueError('Local caption reached max_new_tokens; raise the limit in a new config version.')
            text = self.processor.decode(generated, skip_special_tokens=True)
        if not text or not text.strip():
            raise ValueError('Empty caption; refusal/failure is not a valid description.')
        return text.strip()


In [ ]:
# Cell 6 — Nối manifest của test_attack_v1; không thay đổi hàm sinh caption.
from PIL import Image

samples = [json.loads(line) for line in MANIFEST_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
if not samples:
    raise ValueError("Manifest rỗng. Chạy attack với RUN_ATTACK=True trước.")
ROLES = ("source", "target", "adversarial")
seen_indices = set()
validated_samples = []
for sample in samples:
    if type(sample.get("index")) is not int or sample["index"] in seen_indices:
        raise ValueError("Manifest thiếu index hợp lệ hoặc trùng index.")
    seen_indices.add(sample["index"])
    images, image_sha256 = {}, {}
    for role in ROLES:
        relative = sample.get(role)
        if not isinstance(relative, str) or Path(relative).is_absolute():
            raise ValueError(f"Invalid image path: index={sample['index']}, role={role}")
        path = (INPUT_ROOT / relative).resolve()
        if not path.is_relative_to(INPUT_ROOT) or not path.is_file():
            raise ValueError(f"Unsafe or missing image: {path}")
        # Generator gốc dùng MIME image/png; test_attack_v1 xuất PNG lossless.
        with Image.open(path) as image:
            if image.format != "PNG":
                raise ValueError(f"Input cần là PNG từ test_attack_v1: {path}")
            image.verify()
        digest = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
        images[role] = relative
        image_sha256[role] = digest.hexdigest()
    validated_samples.append({
        **sample, "sample_id": f"{sample['index']:05d}",
        "images": images, "image_sha256": image_sha256,
    })
selected_samples = validated_samples if NUM_SAMPLES is None else validated_samples[:NUM_SAMPLES]
print(f"Validated {len(validated_samples)} samples; selected {len(selected_samples)}.")
print("First sample:", selected_samples[0]["images"])


In [ ]:
# Cell 7 — Sinh captions và ghi từng record; RUN_CAPTION=False không gọi model.
from datetime import datetime, timezone
import shutil
from tqdm.auto import tqdm

RUN_DIR = None
if RUN_CAPTION:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    RUN_DIR = OUTPUT_DIR / f"{CONFIG_NAME}_{stamp}"
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    (RUN_DIR / "config_resolved.yaml").write_text(OmegaConf.to_yaml(cfg, resolve=True), encoding="utf-8")
    config_data = OmegaConf.to_container(cfg, resolve=True)
    fingerprint = hashlib.sha256(json.dumps(config_data, sort_keys=True).encode()).hexdigest()
    run_config = {
        "reference_implementation": REFERENCE_IMPLEMENTATION, "resolved_revision": RESOLVED_REVISION,
        "model": MODEL, "config_name": CONFIG_NAME, "caption_config_sha256": fingerprint,
        "manifest": str(MANIFEST_PATH), "manifest_sha256": hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest(),
        "input_samples": len(validated_samples), "selected_samples": len(selected_samples),
        "num_samples": NUM_SAMPLES, "device": DEVICE, "roles": list(ROLES),
        "python": sys.version, "torch": torch.__version__,
        "caption_source_sha256": hashlib.sha256((SOURCE_DIR / "blackbox_text_generation.py").read_bytes()).hexdigest(),
    }
    (RUN_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2, ensure_ascii=False), encoding="utf-8")
    shutil.copy2(MANIFEST_PATH, RUN_DIR / "manifest.jsonl")
    if (INPUT_ROOT / "run_config.json").is_file():
        shutil.copy2(INPUT_ROOT / "run_config.json", RUN_DIR / "attack_run_config.json")
    # Chỉ snapshot implementation/config; không sao chép secrets hoặc dataset.
    (RUN_DIR / "code").mkdir()
    for filename in ("blackbox_text_generation.py", "utils.py", "config_schema.py"):
        shutil.copy2(SOURCE_DIR / filename, RUN_DIR / "code" / filename)
    shutil.copytree(SOURCE_DIR / "config", RUN_DIR / "code" / "config")
    freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True)
    (RUN_DIR / "requirements-resolved.txt").write_text(freeze.stdout, encoding="utf-8")
    failures, completed_samples, cache = 0, 0, {}
    status = {"status": "running", "samples": 0, "selected_samples": len(selected_samples), "failed_captions": 0}
    (RUN_DIR / "status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
    try:
        generator = VersionedImageDescriptionGenerator(cfg)
        (RUN_DIR / "model_metadata.json").write_text(
            json.dumps({"model_id": generator.model_name, **generator.metadata}, indent=2), encoding="utf-8")
        with (RUN_DIR / "captions.jsonl").open("w", encoding="utf-8") as output:
            for sample in tqdm(selected_samples, desc=generator.model_name):
                row = {
                    **sample, "caption_experiment": cfg.experiment.name,
                    "caption_model": generator.model_name, "caption_config_sha256": fingerprint,
                    "captions": {}, "caption_responses": {}, "errors": {},
                }
                for role in ROLES:
                    checksum = sample["image_sha256"][role]
                    path = INPUT_ROOT / sample["images"][role]
                    try:
                        if hashlib.sha256(path.read_bytes()).hexdigest() != checksum:
                            raise ValueError("Image changed since input validation.")
                        if checksum not in cache:
                            caption = generator.generate_description(str(path))
                            cache[checksum] = (caption, generator.last_response)
                        row["captions"][role], row["caption_responses"][role] = cache[checksum]
                    except Exception as error:
                        row["errors"][role] = type(error).__name__
                        failures += 1
                row["status"] = "ok" if not row["errors"] else "error"
                if row["status"] == "ok":
                    assert set(row["captions"]) == set(ROLES)
                output.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")
                output.flush()
                completed_samples += 1
        status["status"] = "complete" if not failures else "partial"
    except BaseException as error:
        status["status"] = "interrupted" if isinstance(error, KeyboardInterrupt) else "error"
        status["error_type"] = type(error).__name__
        # Giữ thông tin lỗi theo loại, tránh in API credentials từ exception.
        print("Run stopped:", status["error_type"], "| Export artifacts in Cell 8.")
    finally:
        status.update(samples=completed_samples, failed_captions=failures, unique_captions=len(cache))
        (RUN_DIR / "status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
    print("Output:", RUN_DIR, "| Status:", status["status"])
else:
    print("Input/protocol validated. Đặt RUN_CAPTION=True tại Cell 1 rồi chạy lại.")


In [ ]:
# Cell 8 — Preview và ZIP cho Kaggle Output; hỗ trợ export run dở dang.
from IPython.display import display, FileLink
import matplotlib.pyplot as plt

if RUN_DIR is not None and RUN_DIR.is_dir():
    captions_path = RUN_DIR / "captions.jsonl"
    caption_rows = [json.loads(line) for line in captions_path.read_text(encoding="utf-8").splitlines() if line.strip()] if captions_path.is_file() else []
    if caption_rows:
        preview = caption_rows[0]
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        for axis, role in zip(axes, ROLES):
            with Image.open(INPUT_ROOT / preview["images"][role]) as image:
                axis.imshow(image)
            axis.set_title(role)
            axis.axis("off")
        plt.show()
        for role in ROLES:
            print(f"{role}: {preview['captions'].get(role, '[error: ' + preview['errors'].get(role, 'missing') + ']')}")
    ZIP_PATH = Path(shutil.make_archive("/kaggle/working/our_text_generation_v1", "zip", root_dir=RUN_DIR))
    print("Status:", json.loads((RUN_DIR / "status.json").read_text(encoding="utf-8")))
    display(FileLink(str(ZIP_PATH)))
else:
    print("Chưa chạy caption. Đặt RUN_CAPTION=True rồi chạy lại các cell.")
